# Feature Engineering Challenge: Mock Sales Data

In this notebook we start from the **raw CSV file** containing only:

- `Day`: day number in 2025
- `Weather`: cloudy or sunny
- `Sales`: daily sales revenue

Your aim is to create more informative features from the raw data and use them to investigate what seems to influence sales.

This is a small version of the same idea used later in the Rossmann forecasting project: raw data often needs to be transformed before a model or a human can see the important structure.

## 1. Load the raw data

Make sure the CSV file `worksheet1_raw_sales_2025.csv` is in the same folder as this notebook, or update the path below.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

csv_path = "worksheet1_raw_sales_2025.csv"

data = pd.read_csv(csv_path)

data.head()

Check the structure of the dataset.

In [ ]:
data.info()
data.describe(include="all")

## 2. First look at the raw data

Before creating new features, look at the data exactly as given.

Questions:

1. Are sunny days associated with higher sales?
2. Are there unusually high sales values?
3. Can you see a clear pattern from the raw `Day` column alone?

In [ ]:
# Plot sales against day number
plt.figure(figsize=(12, 4))
plt.plot(data["Day"], data["Sales"], marker=".", linewidth=1)
plt.xlabel("Day number in 2025")
plt.ylabel("Sales")
plt.title("Daily sales in the raw data")
plt.show()

In [ ]:
# Compare sales by weather
weather_summary = data.groupby("Weather")["Sales"].agg(["count", "mean", "median", "std", "min", "max"])
weather_summary

In [ ]:
plt.figure(figsize=(6, 4))
data.boxplot(column="Sales", by="Weather")
plt.title("Sales by weather")
plt.suptitle("")
plt.xlabel("Weather")
plt.ylabel("Sales")
plt.show()

## 3. Feature engineering from the day number

The raw file gives the date only as a day number. Since the year is 2025, we can reconstruct the calendar date.

Create a new column called `Date` from the `Day` column.

In [ ]:
# 2025-01-01 is day 1.
# Therefore day n is 2025-01-01 + (n - 1) days.

data["Date"] = pd.to_datetime("2025-01-01") + pd.to_timedelta(data["Day"] - 1, unit="D")

data.head()

Now create calendar features from `Date`.

Complete or run the cell below.

In [ ]:
data["Month"] = data["Date"].dt.month_name().str[:3]
data["MonthNumber"] = data["Date"].dt.month
data["DayOfMonth"] = data["Date"].dt.day
data["Weekday"] = data["Date"].dt.day_name().str[:3]
data["WeekdayNumber"] = data["Date"].dt.weekday  # Monday=0, Sunday=6

data.head(10)

## 4. Does the new representation help?

Now that we have created new features, investigate whether sales are related to the calendar.

Start by comparing sales by weekday.

In [ ]:
weekday_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

weekday_summary = (
    data.groupby("Weekday")["Sales"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .reindex(weekday_order)
)

weekday_summary

In [ ]:
plt.figure(figsize=(8, 4))
data.boxplot(column="Sales", by="Weekday")
plt.title("Sales by weekday")
plt.suptitle("")
plt.xlabel("Weekday")
plt.ylabel("Sales")
plt.show()

Now inspect the largest sales days.

Do they have anything in common?

In [ ]:
data.sort_values("Sales", ascending=False).head(20)[
    ["Day", "Date", "Month", "DayOfMonth", "Weekday", "Weather", "Sales"]
]

## 5. Create your own candidate feature

Look at the high-sales rows above. Try to define a new feature that marks the type of day that seems to have unusually high sales.

Do not worry if your first idea is not perfect. Feature engineering is often iterative.

Examples of possible feature ideas:

- a feature based on weekday;
- a feature based on position within the month;
- a feature combining weekday and position within the month;
- a feature combining weather and calendar information.

Create a new feature below.

In [ ]:
# TODO: Replace this with your own feature definition.
# Example structure:
# data["MyNewFeature"] = ...

# Write your feature here:
data["MyNewFeature"] = False

data[["Day", "Date", "Month", "DayOfMonth", "Weekday", "Weather", "Sales", "MyNewFeature"]].head(20)

Evaluate whether your new feature separates high-sales days from normal days.

In [ ]:
feature_summary = data.groupby("MyNewFeature")["Sales"].agg(["count", "mean", "median", "std", "min", "max"])
feature_summary

In [ ]:
plt.figure(figsize=(6, 4))
data.boxplot(column="Sales", by="MyNewFeature")
plt.title("Sales by engineered feature")
plt.suptitle("")
plt.xlabel("MyNewFeature")
plt.ylabel("Sales")
plt.show()

## 6. Optional: create a stronger calendar feature

If your first feature did not explain the high-sales days well, refine it.

Hints:

- Try combining `Weekday` and `DayOfMonth`.
- Try asking whether some weekdays near the beginning of each month behave differently.
- Compare the days selected by your feature with the top-sales table.

In [ ]:
# TODO: Create a refined engineered feature.
# Example structure only; change the condition.

data["RefinedFeature"] = False

refined_summary = data.groupby("RefinedFeature")["Sales"].agg(["count", "mean", "median", "std", "min", "max"])
refined_summary

In [ ]:
data[data["RefinedFeature"]].sort_values("Date")[[
    "Day", "Date", "Month", "DayOfMonth", "Weekday", "Weather", "Sales"
]]

## 7. Simple predictive comparison

We now compare a simple model using only weather with a model using your engineered feature.

This is not meant to be a sophisticated forecasting model. The point is to test whether the engineered feature contains useful information.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error

# Choose which engineered feature to test.
# Change this to "MyNewFeature" if you prefer.
engineered_feature = "RefinedFeature"

model_data = data.copy()
model_data["IsSunny"] = (model_data["Weather"] == "Sunny").astype(int)
model_data[engineered_feature] = model_data[engineered_feature].astype(int)

# Use a random split here only for this small illustrative exercise.
# In real forecasting, we usually split by time.
train_df, test_df = train_test_split(model_data, test_size=0.25, random_state=42)

# Model A: weather only
X_train_A = train_df[["IsSunny"]]
X_test_A = test_df[["IsSunny"]]

# Model B: weather plus engineered feature
X_train_B = train_df[["IsSunny", engineered_feature]]
X_test_B = test_df[["IsSunny", engineered_feature]]

y_train = train_df["Sales"]
y_test = test_df["Sales"]

model_A = DecisionTreeRegressor(max_depth=3, random_state=42)
model_B = DecisionTreeRegressor(max_depth=3, random_state=42)

model_A.fit(X_train_A, y_train)
model_B.fit(X_train_B, y_train)

pred_A = model_A.predict(X_test_A)
pred_B = model_B.predict(X_test_B)

mae_A = mean_absolute_error(y_test, pred_A)
mae_B = mean_absolute_error(y_test, pred_B)

print("Mean absolute error, weather only:", round(mae_A, 2))
print("Mean absolute error, weather + engineered feature:", round(mae_B, 2))

## 8. Reflection

Answer these questions:

1. Which feature did you create?
2. Why did you choose it?
3. Did it explain the high-sales days?
4. Did it improve prediction error?
5. What does this example teach us about feature engineering?

Write your answers below.

In [ ]:
# Write brief answers as comments.

# 1.
# 2.
# 3.
# 4.
# 5.
